# Volume Profile + CVD Strategy — v13 (Composite Context + Dynamic Value Migration)

v13 redesign based on v12 diagnostics:
- VAH_RECLAIM_LONG disabled (diagnostics only) — was -$3,920, 29% win rate
- Composite profile now used for bias filter (was decorative only in v1-v12)
- Dynamic 60M/180M profiles for value migration detection
- Structure-based stops (replaces fixed 1x ATR)
- Range rotation targets: T1=POC, T2=opposite VA boundary, runner
- Rejection candle quality + 3/5 orderflow confirmation
- Context-based risk multipliers
- Experiment matrix: 11 configs across profile/target/stop/filter axes

Performance optimization: events are aggregated into 1M bars once, then replayed per experiment.

In [ ]:
import io, zipfile, warnings, time, math, collections
from collections import defaultdict, OrderedDict
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dataclasses import dataclass, field
from nautilus_trader.test_kit.providers import TestInstrumentProvider

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi':120,'figure.figsize':(14,5)})
INSTRUMENT = TestInstrumentProvider.btcusdt_perp_binance()
TICK_SIZE, VA_PCT = 0.1, 0.70

PROFILE_DAYS = ['2024-03-04','2024-03-05','2024-03-06','2024-03-07','2024-03-08']
TEST_DAYS    = ['2024-03-11','2024-03-12','2024-03-13','2024-03-14','2024-03-15',
                '2024-03-16','2024-03-17','2024-03-18','2024-03-19','2024-03-20',
                '2024-03-21','2024-03-22','2024-03-23','2024-03-24','2024-03-25',
                '2024-03-26','2024-03-27','2024-03-28']
TEST_DAYS_EXT = TEST_DAYS + ['2024-03-29','2024-03-30']

ACCOUNT_BALANCE  = 100_000.0
MAX_LEVERAGE     = 3.0
MAX_BTC_SIZE     = 5.0
TOUCH_BUFFER     = 0.002
MIN_IMBALANCE    = 1.2
CVD_LOOKBACK = 5
FIRST_NO_TRADE_MIN = 30
LAST_NO_TRADE_MIN  = 15
DAILY_HARD_LOSS    = 0.020
SESSION_CVD_LOOKBACK = 30

EXPERIMENTS = [
    {'id':'A',  'profile_mode':'hybrid',           'target_model':'40_40_20', 'stop_model':'structure',  'filter_set':'composite_migration', 'desc':'Baseline hybrid'},
    {'id':'B1', 'profile_mode':'always_prior',      'target_model':'40_40_20', 'stop_model':'structure',  'filter_set':'composite_migration', 'desc':'Prior-day only'},
    {'id':'B2', 'profile_mode':'always_24h',        'target_model':'40_40_20', 'stop_model':'structure',  'filter_set':'composite_migration', 'desc':'Rolling 24h only'},
    {'id':'B3', 'profile_mode':'hybrid_wide',       'target_model':'40_40_20', 'stop_model':'structure',  'filter_set':'composite_migration', 'desc':'Hybrid wide stale'},
    {'id':'C1', 'profile_mode':'hybrid',            'target_model':'poc_only', 'stop_model':'structure',  'filter_set':'composite_migration', 'desc':'POC-only target'},
    {'id':'C2', 'profile_mode':'hybrid',            'target_model':'50_50',    'stop_model':'structure',  'filter_set':'composite_migration', 'desc':'50/50 split target'},
    {'id':'D1', 'profile_mode':'hybrid',            'target_model':'40_40_20', 'stop_model':'atr_swing',  'filter_set':'composite_migration', 'desc':'ATR swing stop'},
    {'id':'E1', 'profile_mode':'hybrid',            'target_model':'40_40_20', 'stop_model':'structure',  'filter_set':'none',               'desc':'No bias/migration'},
    {'id':'E2', 'profile_mode':'hybrid',            'target_model':'40_40_20', 'stop_model':'structure',  'filter_set':'composite_only',     'desc':'Composite bias only'},
    {'id':'E3', 'profile_mode':'hybrid',            'target_model':'40_40_20', 'stop_model':'structure',  'filter_set':'migration_only',     'desc':'Migration only'},
    {'id':'E4', 'profile_mode':'hybrid',            'target_model':'40_40_20', 'stop_model':'structure',  'filter_set':'composite_migration', 'desc':'Full bias+migration'},
]

print(f'Profile: {PROFILE_DAYS[0]}..{PROFILE_DAYS[-1]}  Test: {TEST_DAYS[0]}..{TEST_DAYS[-1]} ({len(TEST_DAYS)} days)')
print(f'Experiments: {len(EXPERIMENTS)} configs')

In [ ]:
# === Download ===
CACHE = {}
def dl(data_type, date):
    k=(data_type,date)
    if k in CACHE: return CACHE[k]
    url=f'https://data.binance.vision/data/futures/um/daily/{data_type}/BTCUSDT/BTCUSDT-{data_type}-{date}.zip'
    r=requests.get(url); r.raise_for_status()
    CACHE[k]=r.content; return r.content

def rd_agg(date):
    z=zipfile.ZipFile(io.BytesIO(dl('aggTrades',date)))
    c=[n for n in z.namelist() if n.endswith('.csv')][0]
    return pd.read_csv(z.open(c))

def rd_bk(date, sample=50):
    z=zipfile.ZipFile(io.BytesIO(dl('bookTicker',date)))
    c=[n for n in z.namelist() if n.endswith('.csv')][0]
    return pd.read_csv(z.open(c), skiprows=lambda i: i>0 and i%sample!=0)

print('Helper ready.')

In [ ]:
# === Profile Calculator ===
@dataclass
class DayProfile:
    date:str; poc:float; vah:float; val:float
    total_volume:float; price_levels:dict=field(repr=False)

def compute_vp(price_levels, total_volume):
    sp=sorted(price_levels.keys()); cp=max(price_levels,key=price_levels.get)
    pp=sp.index(cp); cv=price_levels[cp]; inc={cp}; l,r=pp-1,pp+1
    while cv<total_volume*VA_PCT and (l>=0 or r<len(sp)):
        vl=price_levels[sp[l]] if l>=0 else 0
        vr=price_levels[sp[r]] if r<len(sp) else 0
        if vl>=vr and l>=0: inc.add(sp[l]); cv+=vl; l-=1
        elif r<len(sp): inc.add(sp[r]); cv+=vr; r+=1
        else: break
    return DayProfile(date='',poc=cp,vah=max(inc),val=min(inc),total_volume=total_volume,price_levels=price_levels)

def calc_profile(df):
    df=df.copy(); df['b']=np.round(df['price']/TICK_SIZE)*TICK_SIZE
    vp=df.groupby('b')['quantity'].sum(); tv=vp.sum(); poc=vp.idxmax()
    levels=vp.sort_index(); idx=list(levels.index); pp=idx.index(poc)
    cv=levels.iloc[pp]; inc={poc}; t=tv*VA_PCT; l,r=pp-1,pp+1
    while cv<t and (l>=0 or r<len(idx)):
        vl=levels.iloc[l] if l>=0 else 0
        vr=levels.iloc[r] if r<len(idx) else 0
        if vl>=vr and l>=0: inc.add(idx[l]); cv+=vl; l-=1
        elif r<len(idx): inc.add(idx[r]); cv+=vr; r+=1
        else: break
    ds=pd.Timestamp(df['transact_time'].iloc[0],unit='ms').strftime('%Y-%m-%d')
    return DayProfile(date=ds,poc=poc,vah=max(inc),val=min(inc),total_volume=tv,price_levels=vp.to_dict())

In [ ]:
# === Data Structures ===
@dataclass
class OneMinuteCandle:
    open_p:float; high:float; low:float; close:float
    buy_vol:float; sell_vol:float; total_vol:float
    delta:float; delta_pct:float
    ts_start:int; ts_end:int
    imb_avg:float=0.0
    volume_by_price:dict=None
    max_price_level_share:float=0.0
    close_location:float=0.0
    body_ratio:float=0.0
    
    def finalize(self):
        rng=self.high-self.low
        self.close_location=(self.close-self.low)/rng if rng>0 else 0.5
        body=abs(self.close-self.open_p)
        self.body_ratio=body/rng if rng>0 else 0
        if self.volume_by_price:
            mx=max(self.volume_by_price.values()) if self.volume_by_price else 0
            self.max_price_level_share=mx/self.total_vol if self.total_vol>0 else 0

class BarBuilder:
    def __init__(self):
        self.bars=[]
        self._cur=None; self._vp={}; self._ts=0
        self._imb_buf=[]
    
    def on_trade(self,price,qty,is_buyer_maker,ts):
        if self._cur is None:
            self._cur={'open':price,'high':price,'low':price,'close':price,
                       'buy':0,'sell':0,'ts':ts}
        self._cur['high']=max(self._cur['high'],price)
        self._cur['low']=min(self._cur['low'],price)
        self._cur['close']=price
        if not is_buyer_maker: self._cur['buy']+=qty
        else: self._cur['sell']+=qty
        b=round(price/TICK_SIZE)*TICK_SIZE
        self._vp[b]=self._vp.get(b,0)+qty
        self._ts=ts
    
    def on_quote(self,bp,ap,bq,aq):
        imb=(bq/aq)*100 if aq>0 and bq>0 else 0
        self._imb_buf.append(imb)
        if len(self._imb_buf)>100: self._imb_buf.pop(0)
    
    def on_minute(self,ts):
        if self._cur is None: return
        c=self._cur
        delta=c['buy']-c['sell']; total=c['buy']+c['sell']
        dr=delta/total if total>0 else 0
        imb_avg=sum(self._imb_buf)/len(self._imb_buf) if self._imb_buf else 0
        candle=OneMinuteCandle(
            open_p=c['open'],high=c['high'],low=c['low'],close=c['close'],
            buy_vol=c['buy'],sell_vol=c['sell'],total_vol=total,
            delta=delta,delta_pct=dr,
            ts_start=c['ts'],ts_end=ts,
            imb_avg=imb_avg,
            volume_by_price=dict(self._vp))
        candle.finalize()
        self.bars.append(candle)
        self._cur=None; self._vp={}
    
    def finalize(self):
        self.on_minute(self._ts)

def build_bars(trades_df, quotes_df):
    bb=BarBuilder()
    events=[]
    for _,r in quotes_df.iterrows():
        events.append({'ts':int(r['transaction_time']),'t':'q',
            'bp':float(r['best_bid_price']),'bq':float(r['best_bid_qty']),
            'ap':float(r['best_ask_price']),'aq':float(r['best_ask_qty'])})
    for _,r in trades_df.iterrows():
        events.append({'ts':int(r['transact_time']),'t':'tr',
            'p':float(r['price']),'q':float(r['quantity']),
            'b':bool(r['is_buyer_maker'])})
    events.sort(key=lambda x:x['ts'])
    last_min=-1
    for ev in events:
        m=ev['ts']//60_000
        if m!=last_min and last_min!=-1:
            bb.on_minute(ev['ts'])
        last_min=m
        if ev['t']=='q': bb.on_quote(ev['bp'],ev['ap'],ev['bq'],ev['aq'])
        else: bb.on_trade(ev['p'],ev['q'],ev['b'],ev['ts'])
    bb.finalize()
    return bb.bars

In [ ]:
# === Composite Bias Engine ===
class CompositeBiasScorer:
    def __init__(self, comp_val, comp_poc, comp_vah):
        self.comp_val=comp_val; self.comp_poc=comp_poc; self.comp_vah=comp_vah
        self.comp_mid=(comp_val+comp_vah)/2
    
    def score(self, current_price, active_poc, active_mid, dyn_60m_poc_slope, dyn_180m_poc_slope, session_cvd):
        s=0
        if current_price>self.comp_poc: s+=1
        else: s-=1
        if active_poc>self.comp_poc: s+=1
        elif active_poc<self.comp_poc: s-=1
        if active_mid>self.comp_mid: s+=1
        elif active_mid<self.comp_mid: s-=1
        if dyn_60m_poc_slope>0: s+=1
        elif dyn_60m_poc_slope<0: s-=1
        if dyn_180m_poc_slope>0: s+=1
        elif dyn_180m_poc_slope<0: s-=1
        if session_cvd=='bullish': s+=1
        elif session_cvd=='bearish': s-=1
        return s
    
    def classify(self, score):
        if score>=3: return 'BULLISH'
        if score>=1: return 'NEUTRAL_BULLISH'
        if score<=-3: return 'BEARISH'
        if score<=-1: return 'NEUTRAL_BEARISH'
        return 'NEUTRAL'

class MigrationDetector:
    def __init__(self, poc_slope_min=0.001):
        self.poc_slope_min=poc_slope_min
        self._prev_60m_poc=None; self._prev_60m_mid=None
        self._prev_180m_poc=None; self._prev_180m_mid=None
    
    def update_prev(self, p60, p180):
        if p60: self._prev_60m_poc=p60.poc; self._prev_60m_mid=p60.midpoint
        if p180: self._prev_180m_poc=p180.poc; self._prev_180m_mid=p180.midpoint
    
    def migration_state(self, p60, p180, price, active_poc):
        s60p, s60m = (p60.poc-self._prev_60m_poc, p60.midpoint-self._prev_60m_mid) if p60 and self._prev_60m_poc is not None else (0,0)
        s180p, s180m = (p180.poc-self._prev_180m_poc, p180.midpoint-self._prev_180m_mid) if p180 and self._prev_180m_poc is not None else (0,0)
        th=self.poc_slope_min
        
        bullish_60=(s60p>th or s60m>th)
        bearish_60=(s60p<-th or s60m<-th)
        
        if bullish_60 and s180p>=0 and price>=active_poc*0.998:
            return 'BULLISH', s60p, s60m, s180p
        if bearish_60 and s180p<=0 and price<=active_poc*1.002:
            return 'BEARISH', s60p, s60m, s180p
        return 'NO_CLEAR', s60p, s60m, s180p

class CandleBuffer:
    def __init__(self):
        self.candles=[]
        self._60m_cache=None; self._180m_cache=None
    
    def push(self, c):
        self.candles.append(c)
        self._60m_cache=None; self._180m_cache=None
    
    def _merge_vp(self, n):
        vp={}
        for c in self.candles[-n:]:
            if c.volume_by_price:
                for px,q in c.volume_by_price.items():
                    vp[px]=vp.get(px,0)+q
        return vp, sum(c.total_vol for c in self.candles[-n:])
    
    def get_60m_profile(self):
        if self._60m_cache is None or len(self.candles)<60: return None
        vp,tv=self._merge_vp(60)
        if len(vp)==0: return None
        self._60m_cache=compute_vp(vp,tv)
        return self._60m_cache
    
    def get_180m_profile(self):
        if self._180m_cache is None or len(self.candles)<180: return None
        vp,tv=self._merge_vp(180)
        if len(vp)==0: return None
        self._180m_cache=compute_vp(vp,tv)
        return self._180m_cache
    
    def clear_old(self):
        while len(self.candles)>180: self.candles.pop(0)

class OrderflowConfirmer:
    @staticmethod
    def cvd_failed_recovery(cvd_hist, n=5, back=15):
        if len(cvd_hist)<back+n: return False, False
        recent=cvd_hist[-back:]
        bullish=(min(recent)>min(cvd_hist[-(back+n):-back]) and cvd_hist[-1]>cvd_hist[-n])
        bearish=(max(recent)<max(cvd_hist[-(back+n):-back]) and cvd_hist[-1]<cvd_hist[-n])
        return bullish, bearish
    
    @staticmethod
    def delta_ok(delta_pcts, bullish=True, n=3):
        if len(delta_pcts)==0: return False
        cur=delta_pcts[-1]
        avg=sum(delta_pcts[-n:])/n if len(delta_pcts)>=n else cur
        if bullish: return cur>=0.0010 or avg>=0.0005
        return cur<=-0.0010 or avg<=-0.0005
    
    @staticmethod
    def imbalance_ok(imb_buf, buy=True, n=3, threshold=1.2):
        vals=[imb_buf[-i] for i in range(1,min(n+1,len(imb_buf)+1)) if len(imb_buf)>=i]
        if not vals: return False
        return sum(vals)/len(vals)>=threshold
    
    @staticmethod
    def absorption_reversal(prices, cvd_hist, lookback=5):
        if len(prices)<lookback+1 or len(cvd_hist)<lookback+1: return False, False
        bullish=(prices[-1]>max(prices[-(lookback+1):-1]) and cvd_hist[-1]<=max(cvd_hist[-(lookback+1):-1]))
        bearish=(prices[-1]<min(prices[-(lookback+1):-1]) and cvd_hist[-1]>=min(cvd_hist[-(lookback+1):-1]))
        return bullish, bearish
    
    @staticmethod
    def volume_ok(volumes, mult=1.0):
        if len(volumes)<20: return False
        return volumes[-1]>=np.median(volumes[-20:])*mult

class StructureStopCalc:
    @staticmethod
    def long_stop(entry, active_val, rejection_low, atr):
        stop_buffer=max(entry*0.0005, atr*0.25)
        structure_stop=min(rejection_low, active_val)-stop_buffer
        min_dist=max(entry*0.0015, atr*0.50)
        max_dist=max(entry*0.0080, atr*1.50)
        dist=entry-structure_stop
        if dist<min_dist: structure_stop=entry-min_dist; dist=min_dist
        return structure_stop, dist, dist>max_dist
    
    @staticmethod
    def short_stop(entry, active_vah, rejection_high, atr):
        stop_buffer=max(entry*0.0005, atr*0.25)
        structure_stop=max(rejection_high, active_vah)+stop_buffer
        min_dist=max(entry*0.0015, atr*0.50)
        max_dist=max(entry*0.0080, atr*1.50)
        dist=structure_stop-entry
        if dist<min_dist: structure_stop=entry+min_dist; dist=min_dist
        return structure_stop, dist, dist>max_dist

class RiskManager:
    def __init__(self, base_risk=0.0075):
        self.base_risk=base_risk
    
    def context_multiplier(self, setup, composite_state, migration_state):
        if setup=='VAL_REJECTION_LONG':
            if composite_state in ('BULLISH','NEUTRAL_BULLISH'): return 1.0
            if composite_state=='NEUTRAL': return 0.75
            if composite_state=='NEUTRAL_BEARISH' and migration_state!='BEARISH': return 0.5
            return 0.0
        elif setup=='VAH_REJECTION_SHORT':
            if composite_state in ('BEARISH','NEUTRAL_BEARISH'): return 1.0
            if composite_state=='NEUTRAL': return 0.75
            if composite_state=='NEUTRAL_BULLISH' and migration_state!='BULLISH': return 0.5
            return 0.0
        return 0.0
    
    def size_position(self, entry, stop, setup, composite_state, migration_state):
        mult=self.context_multiplier(setup, composite_state, migration_state)
        if mult<=0: return 0.0, 0.0, mult
        risk_usdt=ACCOUNT_BALANCE*self.base_risk*mult
        sd=abs(entry-stop)
        if sd<=0: return 0.0, 0.0, mult
        raw=risk_usdt/sd
        max_not=ACCOUNT_BALANCE*MAX_LEVERAGE/entry
        final=min(raw,max_not,MAX_BTC_SIZE)
        if final*sd<0.25*risk_usdt or final<0.001: return 0.0, 0.0, mult
        return final, final*entry/ACCOUNT_BALANCE, mult

class DiagnosticsCollector:
    def __init__(self):
        self.trade_log=[]
        self.funnel=defaultdict(lambda:defaultdict(int))
        self.fail_funnel=defaultdict(lambda:defaultdict(int))
        self.vah_reclaim_diag=[]
    
    def log_funnel(self, setup, stage, passed=True):
        if passed: self.funnel[setup][stage]+=1
        else: self.fail_funnel[setup][stage]+=1
    
    def log_trade(self, **kw): self.trade_log.append(kw)

In [ ]:
# === Strategy V13 ===
class StrategyV13:
    def __init__(self, prior_val, prior_vah, prior_poc, prior_date, prev_agg_df=None,
                 current_date=None, exp_config=None, comp_scorer=None):
        self.prior_val=prior_val; self.prior_vah=prior_vah; self.prior_poc=prior_poc
        self.prior_date=prior_date; self.prev_agg_df=prev_agg_df
        self.current_date=current_date
        cfg=exp_config or {}
        self.pm=cfg.get('profile_mode','hybrid')
        self.tm=cfg.get('target_model','40_40_20')
        self.sm=cfg.get('stop_model','structure')
        self.fs=cfg.get('filter_set','composite_migration')
        self.composite_scorer=comp_scorer
        
        self.day_open=None; self.bar_index=0; self.bar_ts=0
        self.bar_lows=[]; self.bar_highs=[]; self.bar_closes=[]; self.bar_deltas=[]
        self.cvd=0.0; self.cvd_history=[]; self.imb_buf=[]; self.volumes_20=[]
        self.bar_ranges=[]; self.atr=0.0; self.atr_ready=False
        
        self.active_VAL=prior_val; self.active_VAH=prior_vah; self.active_POC=prior_poc
        self.active_source='prior_daily'; self.profile_checked=False
        
        self.trade=None; self.daily_pnl=0.0; self.consecutive_loss_dir={}
        self.last_stop_bar=0; self.last_stop_dir=None
        self.setup=defaultdict(lambda:{'enabled':True,'entries':0,'failures':0})
        self.setup['VAH_RECLAIM_LONG']['enabled']=False
        self.setup['VAL_RETEST_FAILURE_SHORT']['enabled']=False
        
        self.cbuf=CandleBuffer()
        self.migration_detector=MigrationDetector(poc_slope_min=0.001)
        self.orderflow=OrderflowConfirmer()
        self.structure_stop=StructureStopCalc()
        self.risk_mgr=RiskManager(base_risk=0.0075)
        self.diag=DiagnosticsCollector()
        
        self.composite_bias_state='NEUTRAL'
        self.migration_state='NO_CLEAR'
        self.dyn_60m_poc_slope=0; self.dyn_180m_poc_slope=0
    
    def feed_bar(self, candle):
        ts=candle.ts_start
        
        self.cvd+=candle.delta
        self.cvd_history.append(self.cvd)
        self.bar_deltas.append(candle.delta)
        self.bar_lows.append(candle.low)
        self.bar_highs.append(candle.high)
        self.bar_closes.append(candle.close)
        self.volumes_20.append(candle.total_vol)
        self.imb_buf.append(candle.imb_avg)
        
        max_hist=max(CVD_LOOKBACK*2,SESSION_CVD_LOOKBACK*2,30)
        if len(self.bar_lows)>max_hist:
            self.bar_lows.pop(0); self.bar_highs.pop(0); self.bar_closes.pop(0)
            self.cvd_history.pop(0); self.bar_deltas.pop(0)
        if len(self.volumes_20)>20: self.volumes_20.pop(0)
        if len(self.imb_buf)>100: self.imb_buf.pop(0)
        
        rng=candle.high-candle.low
        self.bar_ranges.append(rng)
        if len(self.bar_ranges)>14: self.bar_ranges.pop(0)
        if len(self.bar_ranges)>=14:
            self.atr=sum(self.bar_ranges)/len(self.bar_ranges)
            self.atr_ready=True
        
        self.cbuf.push(candle)
        if len(self.cbuf.candles)>180: self.cbuf.clear_old()
        
        self.bar_index+=1
    
    def _select_active_profile(self):
        if self.profile_checked: return
        self.profile_checked=True
        if self.day_open is None:
            self.day_open=self.bar_closes[0] if self.bar_closes else self.bar_closes[-1] if self.bar_closes else 0
        
        prev=pd.Timestamp(self.prior_date) if isinstance(self.prior_date,str) else pd.Timestamp(str(self.prior_date))
        cur=pd.Timestamp(self.current_date) if isinstance(self.current_date,str) else pd.Timestamp(str(self.current_date))
        weekend=(cur-prev).days>1
        
        open_gap_pct=abs(self.day_open-self.prior_poc)/self.prior_poc if self.prior_poc>0 else 0
        open_gap_atr=open_gap_pct/(self.atr/self.prior_poc) if self.atr>0 and self.prior_poc>0 else 99
        
        stale=False
        if self.pm=='always_24h': stale=True
        elif self.pm=='always_prior': stale=False
        elif self.pm=='hybrid_wide':
            if open_gap_pct>0.010: stale=True
            if open_gap_atr>1.5: stale=True
            if weekend and open_gap_pct>0.010: stale=True
        else:
            if open_gap_pct>0.004: stale=True
            if open_gap_atr>1.0: stale=True
            if weekend and open_gap_pct>0.005: stale=True
        
        if not stale:
            self.active_VAL=self.prior_val; self.active_VAH=self.prior_vah; self.active_POC=self.prior_poc
            self.active_source='prior_daily'; return
        
        if self.prev_agg_df is not None and len(self.prev_agg_df)>1000:
            prof=calc_profile(self.prev_agg_df)
            self.active_VAL=prof.val; self.active_VAH=prof.vah; self.active_POC=prof.poc
            self.active_source='rolling_24h'
    
    def _rejection_ok(self, side, candle):
        if side=='LONG':
            if not (candle.low<=self.active_VAL*(1+0.002)): return False
            if candle.close_location<0.50: return False
            if candle.close<self.active_VAL: return False
            return True
        else:
            if not (candle.high>=self.active_VAH*(1-0.002)): return False
            if candle.close_location>0.50: return False
            if candle.close>self.active_VAH: return False
            return True
    
    def _update_composite_migration(self):
        p60=self.cbuf.get_60m_profile()
        p180=self.cbuf.get_180m_profile()
        self.migration_detector.update_prev(p60, p180)
        ms, s60p, s60m, s180p=self.migration_detector.migration_state(
            p60, p180, self.bar_closes[-1] if self.bar_closes else 0, self.active_POC)
        self.migration_state=ms
        self.dyn_60m_poc_slope=s60p; self.dyn_180m_poc_slope=s180p
        
        if self.composite_scorer and self.bar_closes:
            session_cvd='bullish' if (len(self.cvd_history)>=SESSION_CVD_LOOKBACK and 
                self.cvd_history[-1]>self.cvd_history[-SESSION_CVD_LOOKBACK]) else 'bearish'
            scr=self.composite_scorer.score(
                self.bar_closes[-1], self.active_POC, (self.active_VAL+self.active_VAH)/2,
                s60p, s180p, session_cvd)
            self.composite_bias_state=self.composite_scorer.classify(scr)
    
    def _check_val_rejection_long(self, candle):
        s='VAL_REJECTION_LONG'
        self.diag.log_funnel(s,'near_level',True)
        if not self.setup[s]['enabled'] or self.setup[s]['entries']>=1:
            return self.diag.log_funnel(s,'setup_disabled',False)
        self.diag.log_funnel(s,'setup_enabled',True)
        
        if self.active_source=='stale_no_fallback': return self.diag.log_funnel(s,'profile_invalid',False)
        self.diag.log_funnel(s,'profile_valid',True)
        
        if self.fs in ('composite_migration','composite_only'):
            if self.composite_bias_state=='BEARISH': return self.diag.log_funnel(s,'composite_bias_rejected',False)
        self.diag.log_funnel(s,'composite_bias_ok',True)
        
        if self.fs in ('composite_migration','migration_only'):
            if self.migration_state=='BEARISH' and self.bar_closes[-1]<self.active_POC*0.998:
                return self.diag.log_funnel(s,'value_migration_rejected',False)
        self.diag.log_funnel(s,'value_migration_ok',True)
        
        if not self._rejection_ok('LONG', candle): return self.diag.log_funnel(s,'rejection_candle_invalid',False)
        self.diag.log_funnel(s,'rejection_candle_ok',True)
        
        delta_ok=self.orderflow.delta_ok(self.bar_deltas,bullish=True)
        imb_ok=self.orderflow.imbalance_ok(self.imb_buf,buy=True)
        _, rev_bearish=self.orderflow.absorption_reversal(self.bar_lows,self.cvd_history)
        vol_ok=self.orderflow.volume_ok(self.volumes_20)
        cvd_rec, _=self.orderflow.cvd_failed_recovery(self.cvd_history)
        
        if sum([delta_ok, imb_ok, vol_ok, (not rev_bearish), cvd_rec])<3:
            return self.diag.log_funnel(s,'orderflow_not_confirmed',False)
        self.diag.log_funnel(s,'orderflow_confirmed',True)
        
        if self.sm=='structure':
            rejection_low=min(self.bar_lows[-5:]) if len(self.bar_lows)>=5 else candle.low
            stop_px, sdist, too_wide=self.structure_stop.long_stop(
                candle.close, self.active_VAL, rejection_low, self.atr)
            if too_wide: return self.diag.log_funnel(s,'stop_too_wide',False)
        else:
            buffer=max(candle.close*0.0015,self.atr*0.5)
            swing=min(self.bar_lows[-5:]) if len(self.bar_lows)>=5 else candle.low
            stop_px=min(self.active_VAL-buffer,swing-buffer*0.5)
            if stop_px>=candle.close: return self.diag.log_funnel(s,'stop_invalid',False)
        self.diag.log_funnel(s,'stop_valid',True)
        
        r=candle.close-stop_px
        t1=self.active_POC; t1_dist=abs(t1-candle.close)
        if t1_dist<r*0.8: return self.diag.log_funnel(s,'target_too_close',False)
        t2=self.active_VAH; t2_dist=abs(t2-candle.close)
        if self.tm=='poc_only':
            pass
        elif self.tm=='50_50':
            if t1_dist<r*0.6: return self.diag.log_funnel(s,'target_too_close',False)
        else:
            if t2_dist<r*1.5 and t2_dist>0: return self.diag.log_funnel(s,'t2_target_too_close',False)
        self.diag.log_funnel(s,'target_valid',True)
        
        size, lev, mult=self.risk_mgr.size_position(
            candle.close, stop_px, s, self.composite_bias_state, self.migration_state)
        if size<=0: return self.diag.log_funnel(s,'size_too_small',False)
        self.diag.log_funnel(s,'entered',True)
        
        self._enter(candle.close, s, 'LONG', stop_px, size, r, t1, t2, mult, candle.ts_start)
        return True
    
    def _check_vah_rejection_short(self, candle):
        s='VAH_REJECTION_SHORT'
        self.diag.log_funnel(s,'near_level',True)
        if not self.setup[s]['enabled'] or self.setup[s]['entries']>=1:
            return self.diag.log_funnel(s,'setup_disabled',False)
        self.diag.log_funnel(s,'setup_enabled',True)
        
        if self.active_source=='stale_no_fallback': return self.diag.log_funnel(s,'profile_invalid',False)
        self.diag.log_funnel(s,'profile_valid',True)
        
        if self.fs in ('composite_migration','composite_only'):
            if self.composite_bias_state=='BULLISH': return self.diag.log_funnel(s,'composite_bias_rejected',False)
        self.diag.log_funnel(s,'composite_bias_ok',True)
        
        if self.fs in ('composite_migration','migration_only'):
            if self.migration_state=='BULLISH' and self.bar_closes[-1]>self.active_POC*1.002:
                return self.diag.log_funnel(s,'value_migration_rejected',False)
        self.diag.log_funnel(s,'value_migration_ok',True)
        
        if not self._rejection_ok('SHORT', candle): return self.diag.log_funnel(s,'rejection_candle_invalid',False)
        self.diag.log_funnel(s,'rejection_candle_ok',True)
        
        delta_ok=self.orderflow.delta_ok(self.bar_deltas,bullish=False)
        imb_ok=self.orderflow.imbalance_ok(self.imb_buf,buy=False)
        rev_bullish, _=self.orderflow.absorption_reversal(self.bar_highs,self.cvd_history)
        vol_ok=self.orderflow.volume_ok(self.volumes_20)
        _, cvd_rec_fail=self.orderflow.cvd_failed_recovery(self.cvd_history)
        
        if sum([delta_ok, imb_ok, vol_ok, (not rev_bullish), cvd_rec_fail])<3:
            return self.diag.log_funnel(s,'orderflow_not_confirmed',False)
        self.diag.log_funnel(s,'orderflow_confirmed',True)
        
        if self.sm=='structure':
            rejection_high=max(self.bar_highs[-5:]) if len(self.bar_highs)>=5 else candle.high
            stop_px, sdist, too_wide=self.structure_stop.short_stop(
                candle.close, self.active_VAH, rejection_high, self.atr)
            if too_wide: return self.diag.log_funnel(s,'stop_too_wide',False)
        else:
            buffer=max(candle.close*0.0015,self.atr*0.5)
            swing=max(self.bar_highs[-5:]) if len(self.bar_highs)>=5 else candle.high
            stop_px=max(self.active_VAH+buffer,swing+buffer*0.5)
            if stop_px<=candle.close: return self.diag.log_funnel(s,'stop_invalid',False)
        self.diag.log_funnel(s,'stop_valid',True)
        
        r=stop_px-candle.close
        t1=self.active_POC; t1_dist=abs(candle.close-t1)
        if t1_dist<r*0.8: return self.diag.log_funnel(s,'target_too_close',False)
        t2=self.active_VAL; t2_dist=abs(candle.close-t2)
        if self.tm=='poc_only':
            pass
        elif self.tm=='50_50':
            if t1_dist<r*0.6: return self.diag.log_funnel(s,'target_too_close',False)
        else:
            if t2_dist<r*1.5 and t2_dist>0: return self.diag.log_funnel(s,'t2_target_too_close',False)
        self.diag.log_funnel(s,'target_valid',True)
        
        size, lev, mult=self.risk_mgr.size_position(
            candle.close, stop_px, s, self.composite_bias_state, self.migration_state)
        if size<=0: return self.diag.log_funnel(s,'size_too_small',False)
        self.diag.log_funnel(s,'entered',True)
        
        self._enter(candle.close, s, 'SHORT', stop_px, size, r, t1, t2, mult, candle.ts_start)
        return True
    
    def _check_vah_reclaim_long_diag(self, candle):
        if not candle: return
        self.diag.vah_reclaim_diag.append({
            'bar':self.bar_index,'price':candle.close,
            'above_vah':candle.close>self.active_VAH if self.active_VAH else False,
            'composite_bias':self.composite_bias_state,
            'migration':self.migration_state,
            'volume':candle.total_vol,
            'delta_pct':candle.delta_pct,
            'close_location':candle.close_location,
            'vol_vs_med':candle.total_vol/np.median(self.volumes_20[-20:]) if len(self.volumes_20)>=20 and np.median(self.volumes_20[-20:])>0 else 0
        })
    
    def _enter(self,price,setup,side,stop,size,r,t1,t2,mult,ts):
        self.trade={
            'side':side,'entry_px':price,'entry_ts':ts,
            'stop_px':stop,'total_size':size,'remaining_size':size,
            'setup':setup,'r_val':r,'t1':t1,'t2':t2,
            'target_1_hit':False,'target_2_hit':False,
            'trail_extreme':price,'failure_bars':0,
            'profile_source':self.active_source,'risk_mult':mult
        }
        self.setup[setup]['entries']+=1
        lev=size*price/ACCOUNT_BALANCE
        print(f'>>> {side} {price:.1f} stop={stop:.1f} sz={size:.3f} R={r:.1f} lev={lev:.1f}x ({setup}) src={self.active_source}')
    
    def _check_exit(self, close, candle):
        t=self.trade
        if t is None: return
        if (t['side']=='LONG' and close<=t['stop_px']) or (t['side']=='SHORT' and close>=t['stop_px']):
            self._close(close,'stop',t['remaining_size'],candle)
            self.last_stop_bar=self.bar_index; self.last_stop_dir=t['side']
            self.setup[t['setup']]['failures']+=1
            self.consecutive_loss_dir[t['side']]=self.consecutive_loss_dir.get(t['side'],0)+1
            return
        
        if t['setup'] in ('VAH_REJECTION_SHORT','VAL_REJECTION_LONG'):
            self._exit_range_rotation(close, candle)
        else:
            self._exit_breakout(close, candle)
    
    def _exit_range_rotation(self, close, candle):
        t=self.trade; side=t['setup']
        buffer=max(close*0.0015,self.atr*0.5)
        
        ext_eligible=False
        if t['side']=='LONG':
            ext_eligible=(self.composite_bias_state not in ('BEARISH','NEUTRAL_BEARISH') and self.migration_state!='BEARISH')
        else:
            ext_eligible=(self.composite_bias_state not in ('BULLISH','NEUTRAL_BULLISH') and self.migration_state!='BULLISH')
        
        if t['side']=='LONG':
            if close<self.active_VAL-buffer:
                self._close(close,'failure',t['remaining_size'],candle); return
            if close<self.active_VAL:
                t['failure_bars']+=1
                if t['failure_bars']>=3: self._close(close,'failure',t['remaining_size'],candle); return
            else: t['failure_bars']=0
            
            if not t['target_1_hit'] and close>=t['t1']:
                if self.tm=='poc_only': self._close(close,'poc',t['remaining_size'],candle); return
                sz=t['total_size']*0.4
                self._close(close,'target_1',sz,candle); t['target_1_hit']=True
                t['stop_px']=max(t['stop_px'],self.active_POC-buffer)
            
            if t['target_1_hit'] and t['t2'] and close>=t['t2'] and ext_eligible:
                sz=t['total_size']*0.4
                self._close(close,'target_2',sz,candle); t['target_2_hit']=True
            
            if t['remaining_size']>0:
                t['trail_extreme']=max(t['trail_extreme'],close)
                t['stop_px']=max(t['stop_px'],t['trail_extreme']-max(t['r_val']*0.75,self.atr*1.0))
        else:
            if close>self.active_VAH+buffer:
                self._close(close,'failure',t['remaining_size'],candle); return
            if close>self.active_VAH:
                t['failure_bars']+=1
                if t['failure_bars']>=3: self._close(close,'failure',t['remaining_size'],candle); return
            else: t['failure_bars']=0
            
            if not t['target_1_hit'] and close<=t['t1']:
                if self.tm=='poc_only': self._close(close,'poc',t['remaining_size'],candle); return
                sz=t['total_size']*0.4
                self._close(close,'target_1',sz,candle); t['target_1_hit']=True
                t['stop_px']=min(t['stop_px'],self.active_POC+buffer)
            
            if t['target_1_hit'] and t['t2'] and close<=t['t2'] and ext_eligible:
                sz=t['total_size']*0.4
                self._close(close,'target_2',sz,candle); t['target_2_hit']=True
            
            if t['remaining_size']>0:
                t['trail_extreme']=min(t['trail_extreme'],close)
                t['stop_px']=min(t['stop_px'],t['trail_extreme']+max(t['r_val']*0.75,self.atr*1.0))
    
    def _exit_breakout(self, close, candle):
        t=self.trade; buffer=max(close*0.0015,self.atr*0.5)
        if t['side']=='LONG':
            if close<self.active_VAH-buffer: self._close(close,'failure',t['remaining_size'],candle); return
            if not t['target_1_hit'] and close>=t['t1']:
                sz=t['total_size']*0.5; self._close(close,'target_1',sz,candle); t['target_1_hit']=True
                t['stop_px']=max(t['stop_px'],self.active_VAH)
            if t['remaining_size']>0:
                t['trail_extreme']=max(t['trail_extreme'],close)
                t['stop_px']=max(t['stop_px'],t['trail_extreme']-max(t['r_val']*0.75,self.atr*1.0))
        else:
            if close>self.active_VAH+buffer: self._close(close,'failure',t['remaining_size'],candle); return
            if not t['target_1_hit'] and close<=t['t1']:
                sz=t['total_size']*0.5; self._close(close,'target_1',sz,candle); t['target_1_hit']=True
                t['stop_px']=min(t['stop_px'],self.active_VAL)
            if t['remaining_size']>0:
                t['trail_extreme']=min(t['trail_extreme'],close)
                t['stop_px']=min(t['stop_px'],t['trail_extreme']+max(t['r_val']*0.75,self.atr*1.0))
    
    def _close(self, price, reason, size, candle):
        t=self.trade; side=t['side']
        if size==0: return
        pnl=(price-t['entry_px'])/t['entry_px'] if side=='LONG' else (t['entry_px']-price)/t['entry_px']
        pnl_usd=pnl*size*price
        self.daily_pnl+=pnl_usd
        lev=size*price/ACCOUNT_BALANCE
        print(f'  {reason} {side} {price:.1f} ({pnl*100:.2f}% ${pnl_usd:+.0f}) sz={size:.2f} lev={lev:.1f}x')
        self.diag.log_trade(
            side=side,entry=t['entry_px'],exit=price,
            pnl_pct=pnl,size=size,reason=reason,
            setup=t['setup'],r_val=t['r_val'],leverage=lev,pnl_usd=pnl_usd,
            profile_source=t.get('profile_source','?'),exit_ts=candle.ts_start if candle else 0
        )
        t['remaining_size']-=size
        if t['remaining_size']<=0.001: self.trade=None
    
    def force_close(self, price, candle):
        if self.trade is not None:
            self._close(price,'eod',self.trade['remaining_size'],candle)
            self.trade=None
    
    def process_bar(self, candle):
        if self.day_open is None:
            self.day_open=candle.open_p
        
        if self.bar_index==0:
            self._select_active_profile()
        
        self.feed_bar(candle)
        
        if self.trade is not None:
            self._check_exit(candle.close, candle)
            return
        
        if self.daily_pnl<=-DAILY_HARD_LOSS*ACCOUNT_BALANCE: return
        
        if self.atr_ready:
            self._update_composite_migration()
            self._check_val_rejection_long(candle)
            self._check_vah_rejection_short(candle)
            self._check_vah_reclaim_long_diag(candle)
    
    def get_results(self):
        return self.diag, self.trade is None

In [ ]:
# === Composite Profile ===
t0=time.time()
comp_vp={}; comp_tv=0; day_profs={}
for d in PROFILE_DAYS:
    print(f'Profile {d}...',end=' ')
    df=rd_agg(d); p=calc_profile(df)
    day_profs[d]=p; comp_tv+=p.total_volume
    for px,v in p.price_levels.items(): comp_vp[px]=comp_vp.get(px,0)+v
    print(f'{len(df):,}t VAL={p.val:.0f} POC={p.poc:.0f} VAH={p.vah:.0f}'); del df

COMP=compute_vp(comp_vp,comp_tv)
print(f'Composite: VAL={COMP.val:.0f} POC={COMP.poc:.0f} VAH={COMP.vah:.0f} ({time.time()-t0:.0f}s)')

fig,ax=plt.subplots(figsize=(8,8))
sp=sorted(comp_vp.keys()); vl=[comp_vp[px] for px in sp]
cl=['#e74c3c' if px==COMP.poc else '#f1c40f' if COMP.val<=px<=COMP.vah else '#bdc3c7' for px in sp]
ax.barh(sp,vl,height=0.08,color=cl,ec='none')
ax.axhline(COMP.poc,color='red',lw=2,ls='--',label=f'POC {COMP.poc:.0f}')
ax.axhline(COMP.val,color='orange',lw=2,ls=':',label=f'VAL {COMP.val:.0f}')
ax.axhline(COMP.vah,color='orange',lw=2,ls=':',label=f'VAH {COMP.vah:.0f}')
ax.invert_yaxis(); ax.legend(); ax.set_xlabel('Volume'); ax.set_ylabel('Price')
ax.set_title(f'Composite ({PROFILE_DAYS[0]}..{PROFILE_DAYS[-1]})',fontweight='bold'); ax.grid(alpha=0.3,axis='x')
plt.tight_layout(); plt.show()
for d in PROFILE_DAYS:
    p=day_profs[d]; print(f'  {d}: VAL={p.val:.0f} POC={p.poc:.0f} VAH={p.vah:.0f}')

In [ ]:
# === Pre-process all test days into 1M bars ===
t0=time.time()
first_prior_agg = rd_agg('2024-03-10')
print(f'Pre-cached 2024-03-10 aggTrades for day 1 fallback: {len(first_prior_agg):,} rows')

ALL_BARS={}
test_day_profiles={}
prev_agg_cache={}
prev_agg_cache['2024-03-10']=first_prior_agg

for i, d in enumerate(TEST_DAYS):
    print(f'\n=== {d} ===')
    td=rd_agg(d); bd=rd_bk(d)
    print(f'  aggTrades: {len(td):,} | bookTicker: {len(bd):,}')
    
    prof=calc_profile(td)
    test_day_profiles[d]=prof
    print(f'  Daily profile: VAL={prof.val:.0f} POC={prof.poc:.0f} VAH={prof.vah:.0f}')
    
    bars=build_bars(td, bd)
    ALL_BARS[d]=bars
    print(f'  Built {len(bars)} 1M bars')
    
    prev_agg_cache[d]=td
    del td, bd

print(f'\nPre-processing complete: {time.time()-t0:.0f}s')

In [ ]:
# === Execute All Experiments ===
COMP_SCORER=CompositeBiasScorer(COMP.val,COMP.poc,COMP.vah)
ALL_RESULTS={}
t0=time.time()

for exp in EXPERIMENTS:
    eid=exp['id']
    print(f'\n{"="*70}')
    print(f'  EXPERIMENT V13_{eid}: {exp["desc"]}')
    print(f'  Profile={exp["profile_mode"]} Target={exp["target_model"]} Stop={exp["stop_model"]} Filter={exp["filter_set"]}')
    print(f'{"="*70}')
    
    all_strats=[]
    for i, d in enumerate(TEST_DAYS):
        if i==0:
            prior=day_profs['2024-03-08']
            prior_date='2024-03-08'
            prev_agg=first_prior_agg
        else:
            prior=test_day_profiles[TEST_DAYS[i-1]]
            prior_date=TEST_DAYS[i-1]
            prev_agg=prev_agg_cache.get(TEST_DAYS[i-1])
        
        strat=StrategyV13(prior.val, prior.vah, prior.poc, prior_date, prev_agg,
                          current_date=d, exp_config=exp, comp_scorer=COMP_SCORER)
        
        for candle in ALL_BARS[d]:
            strat.process_bar(candle)
        
        strat.force_close(ALL_BARS[d][-1].close if ALL_BARS[d] else 0, ALL_BARS[d][-1] if ALL_BARS[d] else None)
        all_strats.append(strat)
    
    ALL_RESULTS[eid]={
        'strats':all_strats,
        'config':exp
    }
    print(f'  Experiment done: {sum(len(s.diag.trade_log) for s in all_strats)} total legs')

print(f'\nAll experiments complete: {time.time()-t0:.0f}s')

In [ ]:
# === Cumulative Results ===
for eid, res in ALL_RESULTS.items():
    config=res['config']
    strats=res['strats']
    
    all_tr=[]
    for strat in strats:
        for t in strat.diag.trade_log:
            all_tr.append(t)
    
    if not all_tr:
        print(f'\nV13_{eid} ({config["desc"]}): 0 trades')
        continue
    
    df=pd.DataFrame(all_tr)
    total_usd=df['pnl_usd'].sum()
    wins=df[df['pnl_usd']>0]
    losses=df[df['pnl_usd']<=0]
    
    print(f'\n{"="*75}')
    print(f'  V13_{eid}: {config["desc"]}')
    print(f'  Profile={config["profile_mode"]} Target={config["target_model"]} Stop={config["stop_model"]} Filter={config["filter_set"]}')
    print(f'  Test: {TEST_DAYS[0]} to {TEST_DAYS[-1]} ({len(TEST_DAYS)} days)')
    print(f'{"="*75}')
    print(f'  Total legs: {len(df)}')
    print(f'  Account return: {total_usd/ACCOUNT_BALANCE*100:+.2f}% (${total_usd:+.0f})')
    print(f'  Wins: {len(wins)}/{len(df)} ({len(wins)/len(df)*100:.0f}%)')
    if len(wins)>0: print(f'  Avg win: {wins["pnl_pct"].mean()*100:.2f}%')
    if len(losses)>0: print(f'  Avg loss: {losses["pnl_pct"].mean()*100:.2f}%')
    if 'leverage' in df.columns:
        print(f'  Avg lev: {df["leverage"].mean():.2f}x  Max lev: {df["leverage"].max():.2f}x')
    if len(losses)>0: print(f'  Max loss: ${losses["pnl_usd"].min():.0f}')
    if len(wins)>0: print(f'  Max win: ${wins["pnl_usd"].max():.0f}')
    
    for setup in ['VAL_REJECTION_LONG','VAH_REJECTION_SHORT']:
        sub=df[df['setup']==setup]
        if len(sub)>0:
            sw=sub[sub['pnl_usd']>0]
            print(f'  {setup}: {len(sub)} legs ${sub["pnl_usd"].sum():+.0f} ({len(sw)}/{len(sub)} wins)')

In [ ]:
# === Comparison Table ===
print(f'\n{"="*75}')
print(f'  EXPERIMENT COMPARISON')
print(f'{"="*75}')
print(f'{"Exp":6s} {"Desc":25s} {"Legs":5s} {"PnL%":8s} {"PnL$":10s} {"Win%":6s} {"AvgLv":6s}')
print(f'{"-"*6} {"-"*25} {"-"*5} {"-"*8} {"-"*10} {"-"*6} {"-"*6}')

rows=[]
for eid, res in ALL_RESULTS.items():
    config=res['config']
    strats=res['strats']
    all_tr=[]
    for strat in strats:
        for t in strat.diag.trade_log: all_tr.append(t)
    
    if all_tr:
        df=pd.DataFrame(all_tr)
        total_usd=df['pnl_usd'].sum()
        wins=df[df['pnl_usd']>0]
        wr=len(wins)/len(df)*100 if len(df)>0 else 0
        avg_lev=df['leverage'].mean() if 'leverage' in df.columns else 0
        pnl_pct=total_usd/ACCOUNT_BALANCE*100
        print(f'{eid:<6s} {config["desc"]:<25s} {len(df):<5d} {pnl_pct:>+7.2f}% {total_usd:>+9.0f} {wr:>5.0f}% {avg_lev:>5.2f}x')
        rows.append({'exp':f'V13_{eid}','desc':config['desc'],'legs':len(df),'pnl_pct':pnl_pct,'pnl_usd':total_usd,'wr':wr,'avg_lev':avg_lev})
    else:
        print(f'{eid:<6s} {config["desc"]:<25s} {"0":>5s} {"0.00%":>8s} {"$0":>10s} {"0%":>6s} {"0x":>6s}')

print(f'\n{"="*75}')
print(f'  FUNNEL DIAGNOSTICS - V13_A (Baseline)')
print(f'{"="*75}')
baseline=ALL_RESULTS.get('A')
if baseline:
    stages=['near_level','setup_enabled','profile_valid','composite_bias_ok','value_migration_ok',
            'rejection_candle_ok','orderflow_confirmed','stop_valid','target_valid','entered']
    for sn in ['VAL_REJECTION_LONG','VAH_REJECTION_SHORT']:
        cf=defaultdict(int); ff=defaultdict(int)
        for s in baseline['strats']:
            for k,v in s.diag.funnel.get(sn,{}).items(): cf[k]+=v
            for k,v in s.diag.fail_funnel.get(sn,{}).items(): ff[k]+=v
        total=cf.get('near_level',0)
        if total==0: continue
        print(f'\n  {sn}: {total} candidates')
        for st in stages:
            cnt=cf.get(st,0)
            print(f'    {" >>>" if st=="entered" else "    "} {cnt:6d}/{total} ({cnt/total*100:5.1f}%)  {st}')
        fail_items=sorted([(k,v) for k,v in ff.items()],key=lambda x:-x[1])[:5]
        if fail_items:
            print(f'    Top fails:')
            for k,v in fail_items:
                print(f'      {v:6d} ({v/total*100:5.1f}%)  {k}')

In [ ]:
# === VAH_RECLAIM Diagnostic Summary ===
baseline=ALL_RESULTS.get('A')
if baseline:
    all_diag=[]
    for s in baseline['strats']:
        all_diag.extend(s.diag.vah_reclaim_diag)
    print(f'\nVAH_RECLAIM DIAGNOSTICS ({len(all_diag)} total bars)')
    above=[d for d in all_diag if d.get('above_vah')]
    print(f'  Bars above VAH: {len(above)}')
    if above:
        bc=sum(1 for d in above if d.get('composite_bias') in ('BULLISH','NEUTRAL_BULLISH'))
        vs=sum(1 for d in above if d.get('vol_vs_med',0)>=1.5)
        hd=sum(1 for d in above if d.get('delta_pct',0)>=0.0020)
        hc=sum(1 for d in above if d.get('close_location',0)>=0.75)
        print(f'  Bullish/neutral-bullish composite: {bc}')
        print(f'  Volume >=1.5x median: {vs}')
        print(f'  Delta >=0.20%: {hd}')
        print(f'  Close loc >=0.75: {hc}')

In [ ]:
# === Charts - Baseline First 5 Days ===
baseline=ALL_RESULTS.get('A')
if baseline:
    for i, d in enumerate(TEST_DAYS[:5]):
        strat=baseline['strats'][i]
        bars=ALL_BARS[d]
        if not bars: continue
        
        fig,ax=plt.subplots(figsize=(16,5))
        times=[pd.Timestamp(b.ts_start,unit='ms') for b in bars]
        prices=[b.open_p for b in bars]
        ax.plot(times,prices,'k-',lw=0.5,alpha=0.7)
        
        for label,px,color in [
            (f'Comp VAL {COMP.val:.0f}',COMP.val,'gold'),
            (f'Comp POC {COMP.poc:.0f}',COMP.poc,'red'),
            (f'Comp VAH {COMP.vah:.0f}',COMP.vah,'gold'),
        ]:
            ax.axhline(px,color=color,lw=1,ls=':',alpha=0.5)
        
        for label,px,color in [
            (f'Active VAL {strat.active_VAL:.0f}',strat.active_VAL,'orange'),
            (f'Active POC {strat.active_POC:.0f}',strat.active_POC,'lime'),
            (f'Active VAH {strat.active_VAH:.0f}',strat.active_VAH,'orange'),
        ]:
            ax.axhline(px,color=color,lw=1.5,ls='--',alpha=0.8)
        
        for t in strat.diag.trade_log:
            col='green' if t.get('pnl_usd',0)>0 else 'red'
            mk='^' if t['side']=='LONG' else 'v'
            ax.scatter(pd.Timestamp(t.get('entry_ts',0),unit='ms'),t['entry'],
                      marker=mk,color=col,s=150,zorder=5,edgecolors='black',linewidth=0.8)
        
        ax.set_title(f'{d} | {len(strat.diag.trade_log)} trades | src={strat.active_source} | bias={strat.composite_bias_state}',
                     fontweight='bold')
        ax.set_ylabel('Price'); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()

---
## v13 Changes

- **VAH_RECLAIM_LONG disabled** (diagnostics only) — was -$3,920, 29% win rate
- **Composite bias filter** — +3/-3 scoring using price vs POC, active vs composite POC, dynamic POC slopes, CVD
- **Dynamic 60M/180M value migration** — POC and midpoint slopes recomputed every 5/15 min from 1M bar aggregates
- **Structure-based stops** — replaces fixed 1x ATR; identifies buyer/seller interest zone, min/max distance checks
- **Range rotation targets** — T1=POC (40%), T2=opposite VA (40%), runner (20%) with extension eligibility checks
- **Rejection candle quality** — close_location, body_ratio, max_price_level_volume_share
- **3/5 orderflow confirmation** — delta, imbalance, CVD recovery/failure, absorption, volume
- **Context-based risk multipliers** — composite + migration determines 1.0x/0.75x/0.50x/0.00x
- **Experiment matrix** — 11 configs across profile mode, target model, stop model, filter set sensitivity
- **Performance optimization**: Events → 1M bars once, replayed per experiment